# Biohub Cell Tracking — Submission Notebook

Self-contained inference notebook for the [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development) Kaggle competition.

**Pipeline**: anisotropic Gaussian smoothing → percentile threshold → connected components → Hungarian frame-to-frame linking → division detection (second-daughter promotion).

**Local evaluation** (`metric.py`) should be run separately against `train/` before submitting — it is intentionally *not* included in this notebook.

## Imports

In [1]:
import json
import os
import time
import traceback

import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# ---------------------------------------------------------------------------
# Chunk decompressor: prefer numcodecs (Blosc wrapper), fall back to raw
# blosc2 if numcodecs is unavailable in the Kaggle environment.
# ---------------------------------------------------------------------------
try:
    from numcodecs import Blosc as _NumcodecsBlosc
    _USE_NUMCODECS = True
    print("Using numcodecs.Blosc for chunk decompression.")
except ImportError:
    _USE_NUMCODECS = False
    import blosc2
    print("numcodecs not available — falling back to blosc2.decompress().")

Using numcodecs.Blosc for chunk decompression.


## Configuration

In [9]:
# Kaggle environment paths
# TEST_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"
# OUTPUT_PATH = "/kaggle/working/submission.csv"

TEST_DIR = r"biohub-cell-tracking-during-development\test"
OUTPUT_PATH = r"submission.csv"

# Physical voxel scale in micrometers (from competition spec)
VOXEL_SCALE = np.array([1.625, 0.40625, 0.40625])  # z, y, x

# Detector parameters (detect_cells_threshold)
DETECTION_SIGMA_UM = 2.0         # Gaussian smoothing sigma in µm
DETECTION_THRESHOLD_PCT = 85.0   # Percentile threshold for foreground
DETECTION_MIN_SIZE = 10          # Minimum connected-component size (voxels)

# Tracker parameters
MAX_LINK_DISTANCE_UM = 7.0       # Max Hungarian matching distance in µm

## Data I/O

Low-level Zarr v3 chunk reader. Each chunk is one timepoint, Blosc-compressed
(zstd + bitshuffle). We read the raw bytes and decompress ourselves because
`zarr-python` 2.x doesn’t support v3 natively.

In [10]:
def get_array_meta(zarr_path: str) -> dict:
    """Read the array-level zarr.json at <zarr>/0/zarr.json."""
    with open(os.path.join(zarr_path, "0", "zarr.json"), "r") as f:
        return json.load(f)


def load_volume_frame(zarr_path: str, t: int) -> np.ndarray:
    """Load a single timepoint from a Zarr v3 image volume.

    Returns
    -------
    np.ndarray
        3-D array of shape (Z, Y, X), dtype uint16.
    """
    meta = get_array_meta(zarr_path)
    shape = tuple(meta["shape"])        # (T, Z, Y, X)
    chunk_shape = tuple(meta["chunk_grid"]["configuration"]["chunk_shape"])
    dtype = np.dtype(meta["data_type"]).newbyteorder("<")

    if t < 0 or t >= shape[0]:
        raise IndexError(f"Timepoint {t} out of range [0, {shape[0]})")

    chunk_path = os.path.join(zarr_path, "0", "c", str(t), "0", "0", "0")

    with open(chunk_path, "rb") as f:
        raw = f.read()

    # Decompress using whichever backend is available
    if _USE_NUMCODECS:
        codec = _NumcodecsBlosc(cname="zstd", clevel=1, shuffle=_NumcodecsBlosc.BITSHUFFLE)
        buf = codec.decode(raw)
    else:
        buf = blosc2.decompress(raw)

    arr = np.frombuffer(buf, dtype=dtype).reshape(chunk_shape)
    return arr[0]  # remove leading T=1 dimension -> (Z, Y, X)


def voxel_to_physical(coords_zyx: np.ndarray) -> np.ndarray:
    """Convert voxel coordinates to physical (µm) coordinates."""
    return coords_zyx * VOXEL_SCALE

## Detector

**Default**: `detect_cells_threshold` — anisotropic Gaussian smoothing + percentile
threshold + connected components + centre-of-mass. Fast and robust.

> **Note:** `detect_cells` (LoG / `skimage.feature.blob_log`) is available in the
> original modular codebase and generally more accurate, but significantly slower
> per frame due to multi-scale convolution. Benchmark runtime before swapping it
> in for a full test-set submission.

In [11]:
def detect_cells_threshold(
    volume_3d: np.ndarray,
    voxel_scale: np.ndarray = None,
    sigma_um: float = DETECTION_SIGMA_UM,
    threshold_percentile: float = DETECTION_THRESHOLD_PCT,
    min_size_voxels: int = DETECTION_MIN_SIZE,
) -> np.ndarray:
    """Gaussian smooth + threshold + connected components detector.

    Returns
    -------
    np.ndarray, shape (N, 3)
        Centroids in voxel units (z, y, x).
    """
    if voxel_scale is None:
        voxel_scale = VOXEL_SCALE

    # Gaussian smoothing with anisotropic sigma
    sigma_voxels = sigma_um / voxel_scale
    smoothed = ndimage.gaussian_filter(
        volume_3d.astype(np.float32), sigma=sigma_voxels
    )

    # Threshold
    thresh = np.percentile(smoothed, threshold_percentile)
    binary = smoothed > thresh

    # Connected components
    labeled, n_labels = ndimage.label(binary)

    centroids = []
    for i in range(1, n_labels + 1):
        component = labeled == i
        size = component.sum()
        if size < min_size_voxels:
            continue
        # Center of mass
        com = ndimage.center_of_mass(smoothed, labeled, i)
        centroids.append(com)

    if len(centroids) == 0:
        return np.empty((0, 3), dtype=np.float64)

    return np.array(centroids, dtype=np.float64)

## Tracker

Hungarian (one-to-one) frame-to-frame linking in physical (µm) space,
followed by a second pass that promotes unmatched nodes to second daughters
of already-matched parents (division detection).

In [12]:
class TrackGraph:
    """Simple tracking graph container.

    Attributes
    ----------
    nodes : list of dict
        Each dict has keys: 'id', 't', 'z', 'y', 'x' (voxel coords, int).
    edges : list of tuple
        Each tuple is (source_node_id, target_node_id).
    """

    def __init__(self):
        self.nodes = []
        self.edges = []
        self._next_id = 1

    def add_node(self, t: int, z: float, y: float, x: float) -> int:
        """Add a node and return its ID."""
        nid = self._next_id
        self._next_id += 1
        self.nodes.append({
            "id": nid,
            "t": t,
            "z": int(round(z)),
            "y": int(round(y)),
            "x": int(round(x)),
        })
        return nid

    def add_edge(self, source_id: int, target_id: int):
        """Add a directed edge from source to target."""
        self.edges.append((source_id, target_id))

    def get_nodes_at_time(self, t: int) -> list:
        """Return all nodes at timepoint t."""
        return [n for n in self.nodes if n["t"] == t]

    def get_outgoing_edges(self, node_id: int) -> list:
        """Return target IDs for outgoing edges from node_id."""
        return [tgt for src, tgt in self.edges if src == node_id]

    def get_incoming_edges(self, node_id: int) -> list:
        """Return source IDs for incoming edges to node_id."""
        return [src for src, tgt in self.edges if tgt == node_id]


def link_frames(
    centroids_t0: np.ndarray,
    centroids_t1: np.ndarray,
    voxel_scale: np.ndarray = None,
    max_distance_um: float = MAX_LINK_DISTANCE_UM,
) -> list:
    """Link detections between two consecutive frames using Hungarian matching.

    Returns list of (i, j) index pairs where i indexes centroids_t0
    and j indexes centroids_t1. Unmatched detections are not included.
    """
    if voxel_scale is None:
        voxel_scale = VOXEL_SCALE

    if len(centroids_t0) == 0 or len(centroids_t1) == 0:
        return []

    # Convert to physical coordinates
    phys_t0 = voxel_to_physical(centroids_t0)
    phys_t1 = voxel_to_physical(centroids_t1)

    # Compute pairwise distance matrix
    cost = cdist(phys_t0, phys_t1, metric="euclidean")

    # Mask out distances beyond threshold with large sentinel
    SENTINEL = 1e6
    cost_masked = np.where(cost <= max_distance_um, cost, SENTINEL)

    # Hungarian assignment
    row_ind, col_ind = linear_sum_assignment(cost_masked)

    # Filter out assignments that exceed the distance threshold
    matches = []
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] <= max_distance_um:
            matches.append((int(i), int(j)))

    return matches


def build_tracks(
    all_centroids: dict,
    voxel_scale: np.ndarray = None,
    max_distance_um: float = MAX_LINK_DISTANCE_UM,
) -> TrackGraph:
    """Build a full tracking graph from per-frame detections."""
    if voxel_scale is None:
        voxel_scale = VOXEL_SCALE

    graph = TrackGraph()
    timepoints = sorted(all_centroids.keys())

    if len(timepoints) == 0:
        return graph

    # Add nodes for all timepoints, keep track of node IDs per frame
    frame_node_ids = {}  # t -> list of node IDs (same order as centroids)

    for t in timepoints:
        centroids = all_centroids[t]
        ids = []
        for c in centroids:
            nid = graph.add_node(t, c[0], c[1], c[2])
            ids.append(nid)
        frame_node_ids[t] = ids

    # Link consecutive frames
    for i in range(len(timepoints) - 1):
        t0, t1 = timepoints[i], timepoints[i + 1]

        # Only link consecutive timepoints (t1 == t0 + 1)
        if t1 != t0 + 1:
            continue

        centroids_t0 = all_centroids[t0]
        centroids_t1 = all_centroids[t1]
        ids_t0 = frame_node_ids[t0]
        ids_t1 = frame_node_ids[t1]

        matches = link_frames(centroids_t0, centroids_t1, voxel_scale, max_distance_um)

        for idx0, idx1 in matches:
            graph.add_edge(ids_t0[idx0], ids_t1[idx1])

    return graph


def detect_divisions(
    graph: TrackGraph,
    all_centroids: dict,
    voxel_scale: np.ndarray = None,
    max_distance_um: float = MAX_LINK_DISTANCE_UM,
) -> TrackGraph:
    """Detect cell divisions by promoting unmatched nodes to second daughters
    of nearby parents that already have exactly one outgoing edge.

    Modifies graph in place and returns it.
    """
    if voxel_scale is None:
        voxel_scale = VOXEL_SCALE

    timepoints = sorted(all_centroids.keys())

    for i in range(1, len(timepoints)):
        t = timepoints[i]
        t_prev = timepoints[i - 1]

        if t != t_prev + 1:
            continue

        # Find nodes at t with no incoming edges (unmatched / new)
        nodes_t = graph.get_nodes_at_time(t)
        unmatched = [n for n in nodes_t if len(graph.get_incoming_edges(n["id"])) == 0]

        if len(unmatched) == 0:
            continue

        # Get parent nodes at t-1
        nodes_prev = graph.get_nodes_at_time(t_prev)
        if len(nodes_prev) == 0:
            continue

        # Build arrays for distance computation
        parent_coords = np.array([[n["z"], n["y"], n["x"]] for n in nodes_prev], dtype=float)
        parent_ids = [n["id"] for n in nodes_prev]
        parent_phys = voxel_to_physical(parent_coords)

        for node in unmatched:
            child_coord = np.array([[node["z"], node["y"], node["x"]]], dtype=float)
            child_phys = voxel_to_physical(child_coord)

            # Distance to all potential parents
            dists = cdist(child_phys, parent_phys, metric="euclidean")[0]
            nearest_idx = np.argmin(dists)
            nearest_dist = dists[nearest_idx]

            if nearest_dist > max_distance_um:
                continue

            parent_id = parent_ids[nearest_idx]

            # Only create a division if this parent already has exactly 1 outgoing edge
            # (making it a 1->2 split)
            outgoing = graph.get_outgoing_edges(parent_id)
            if len(outgoing) == 1:
                graph.add_edge(parent_id, node["id"])

    return graph

## Submission helpers

In [13]:
def graph_to_submission_rows(graph: TrackGraph, dataset_name: str) -> list:
    """Convert a TrackGraph to submission CSV rows.

    Returns list of dicts with keys matching the submission schema:
    dataset, row_type, node_id, t, z, y, x, source_id, target_id.
    """
    rows = []

    # Node rows
    for node in graph.nodes:
        rows.append({
            "dataset": dataset_name,
            "row_type": "node",
            "node_id": node["id"],
            "t": node["t"],
            "z": node["z"],
            "y": node["y"],
            "x": node["x"],
            "source_id": -1,
            "target_id": -1,
        })

    # Edge rows
    for src, tgt in graph.edges:
        rows.append({
            "dataset": dataset_name,
            "row_type": "edge",
            "node_id": -1,
            "t": -1,
            "z": -1,
            "y": -1,
            "x": -1,
            "source_id": src,
            "target_id": tgt,
        })

    return rows

## Main inference loop

Iterates over every `.zarr` folder in `TEST_DIR`. Each dataset is wrapped in
`try/except` so one corrupt or slow file doesn’t kill the entire submission.
Elapsed time is logged per dataset for Kaggle log diagnostics.

In [14]:
# Discover all test samples
test_folder_names = sorted(
    f.replace(".zarr", "") for f in os.listdir(TEST_DIR) if f.endswith(".zarr")
)
print(f"Found {len(test_folder_names)} test samples: {test_folder_names}")

all_rows = []
failed_datasets = []
pipeline_start = time.time()

for idx, folder_name in enumerate(test_folder_names):
    t_start = time.time()
    zarr_path = os.path.join(TEST_DIR, folder_name + ".zarr")
    print(f"\n[{idx + 1}/{len(test_folder_names)}] {folder_name}")

    try:
        # Read volume metadata
        meta = get_array_meta(zarr_path)
        shape = tuple(meta["shape"])  # (T, Z, Y, X)
        n_t = shape[0]
        print(f"  Shape: {shape}, {n_t} timepoints")

        # --- Detection: per-frame ---
        all_centroids = {}
        for t in range(n_t):
            frame = load_volume_frame(zarr_path, t)
            centroids = detect_cells_threshold(frame, VOXEL_SCALE)
            all_centroids[t] = centroids
            if t % 20 == 0 or t == n_t - 1:
                print(f"    t={t:3d}: {len(centroids):4d} detections")

        # --- Tracking: Hungarian linking ---
        graph = build_tracks(all_centroids, VOXEL_SCALE, MAX_LINK_DISTANCE_UM)

        # --- Division detection ---
        graph = detect_divisions(graph, all_centroids, VOXEL_SCALE, MAX_LINK_DISTANCE_UM)

        n_divs = sum(
            1 for n in graph.nodes if len(graph.get_outgoing_edges(n["id"])) >= 2
        )
        print(
            f"  Result: {len(graph.nodes)} nodes, {len(graph.edges)} edges, "
            f"{n_divs} divisions"
        )

        # --- Collect submission rows ---
        rows = graph_to_submission_rows(graph, folder_name)
        all_rows.extend(rows)

    except Exception:
        print(f"  ERROR processing {folder_name} — skipping.")
        traceback.print_exc()
        failed_datasets.append(folder_name)

    elapsed = time.time() - t_start
    print(f"  Elapsed: {elapsed:.1f}s")

total_elapsed = time.time() - pipeline_start
print(f"\n{'='*60}")
print(f"Pipeline complete. Total time: {total_elapsed:.1f}s")
print(f"Datasets processed: {len(test_folder_names) - len(failed_datasets)}/{len(test_folder_names)}")
if failed_datasets:
    print(f"FAILED datasets: {failed_datasets}")

Found 4 test samples: ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']

[1/4] 44b6_0113de3b
  Shape: (100, 64, 256, 256), 100 timepoints
    t=  0:   10 detections
    t= 20:   21 detections
    t= 40:   25 detections
    t= 60:   37 detections
    t= 80:   45 detections
    t= 99:   26 detections
  Result: 3133 nodes, 2329 edges, 19 divisions
  Elapsed: 155.6s

[2/4] 44b6_0b24845f
  Shape: (100, 64, 256, 256), 100 timepoints
    t=  0:   14 detections
    t= 20:    7 detections
    t= 40:    8 detections
    t= 60:   12 detections
    t= 80:   16 detections
    t= 99:    9 detections
  Result: 1010 nodes, 493 edges, 4 divisions
  Elapsed: 78.6s

[3/4] 6bba_05b6850b
  Shape: (100, 64, 256, 256), 100 timepoints
    t=  0:    2 detections
    t= 20:    2 detections
    t= 40:    2 detections
    t= 60:    2 detections
    t= 80:    2 detections
    t= 99:    1 detections
  Result: 207 nodes, 167 edges, 0 divisions
  Elapsed: 35.7s

[4/4] 6bba_05db0fb1
  Shape: (100, 6

## Submission export

Schema: `id, dataset, row_type, node_id, t, z, y, x, source_id, target_id`.
The `id` column is a single globally-contiguous counter (0-based) across the
whole file — not reset per dataset. The `node_id` counters *do* reset per
dataset, matching baseline convention.

In [15]:
submission = pd.DataFrame(all_rows)
submission.insert(0, "id", range(len(submission)))
submission = submission[
    ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
]
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Submission written: {OUTPUT_PATH} ({len(submission)} rows)")
print(submission.head(10))

Submission written: submission.csv (14159 rows)
   id        dataset row_type  node_id  t   z    y    x  source_id  target_id
0   0  44b6_0113de3b     node        1  0  34  131  143         -1         -1
1   1  44b6_0113de3b     node        2  0  15  201  106         -1         -1
2   2  44b6_0113de3b     node        3  0  21  123  158         -1         -1
3   3  44b6_0113de3b     node        4  0  23   45  180         -1         -1
4   4  44b6_0113de3b     node        5  0  26   91  162         -1         -1
5   5  44b6_0113de3b     node        6  0  32  137  187         -1         -1
6   6  44b6_0113de3b     node        7  0  34    6  224         -1         -1
7   7  44b6_0113de3b     node        8  0  39   96  217         -1         -1
8   8  44b6_0113de3b     node        9  0  46   19  217         -1         -1
9   9  44b6_0113de3b     node       10  0  50  189  250         -1         -1


## Verification

Reload `submission.csv` and run sanity checks before hitting Submit.

In [16]:
# Reload from disk to verify what was actually written
df = pd.read_csv(OUTPUT_PATH)
print(f"Loaded {len(df)} rows from {OUTPUT_PATH}")
print(f"Columns: {list(df.columns)}")
print()

# 1. Every .zarr folder appears at least once
expected_datasets = set(
    f.replace(".zarr", "") for f in os.listdir(TEST_DIR) if f.endswith(".zarr")
)
actual_datasets = set(df["dataset"].unique())
missing = expected_datasets - actual_datasets
assert len(missing) == 0, f"Missing datasets in submission: {missing}"
print(f"✓ All {len(expected_datasets)} test datasets present in submission.")

# 2. No duplicate (dataset, node_id) among node rows
node_rows = df[df["row_type"] == "node"]
dup_check = node_rows.groupby(["dataset", "node_id"]).size()
duplicates = dup_check[dup_check > 1]
assert len(duplicates) == 0, f"Duplicate (dataset, node_id) pairs:\n{duplicates}"
print(f"✓ No duplicate (dataset, node_id) pairs among {len(node_rows)} node rows.")

# 3. Every edge's source_id/target_id refers to a valid node in the same dataset
edge_rows = df[df["row_type"] == "edge"]
all_ok = True
for ds in actual_datasets:
    ds_nodes = set(node_rows[node_rows["dataset"] == ds]["node_id"])
    ds_edges = edge_rows[edge_rows["dataset"] == ds]
    bad_src = ds_edges[~ds_edges["source_id"].isin(ds_nodes)]
    bad_tgt = ds_edges[~ds_edges["target_id"].isin(ds_nodes)]
    if len(bad_src) > 0 or len(bad_tgt) > 0:
        print(f"✗ {ds}: {len(bad_src)} dangling source_ids, {len(bad_tgt)} dangling target_ids")
        all_ok = False
if all_ok:
    print(f"✓ All edge source_id/target_id references are valid.")

# 4. Per-dataset summary table
print(f"\n{'='*70}")
print(f"{'Dataset':<40} {'Nodes':>8} {'Edges':>8} {'Divisions':>10}")
print(f"{'-'*70}")
for ds in sorted(actual_datasets):
    n_nodes = len(node_rows[node_rows["dataset"] == ds])
    n_edges = len(edge_rows[edge_rows["dataset"] == ds])
    # Count divisions: nodes whose node_id appears as source_id in >=2 edges
    ds_edge_df = edge_rows[edge_rows["dataset"] == ds]
    src_counts = ds_edge_df["source_id"].value_counts()
    n_divs = int((src_counts >= 2).sum())
    print(f"{ds:<40} {n_nodes:>8} {n_edges:>8} {n_divs:>10}")
print(f"{'-'*70}")
print(f"{'TOTAL':<40} {len(node_rows):>8} {len(edge_rows):>8}")
print(f"\n✓ Verification complete.")

Loaded 14159 rows from submission.csv
Columns: ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']

✓ All 4 test datasets present in submission.
✓ No duplicate (dataset, node_id) pairs among 8457 node rows.
✓ All edge source_id/target_id references are valid.

Dataset                                     Nodes    Edges  Divisions
----------------------------------------------------------------------
44b6_0113de3b                                3133     2329         19
44b6_0b24845f                                1010      493          4
6bba_05b6850b                                 207      167          0
6bba_05db0fb1                                4107     2713         69
----------------------------------------------------------------------
TOTAL                                        8457     5702

✓ Verification complete.
